# Artificial Neural Networks (ANNs)

## Perceptron

- one of the simplest ANN architecture.
- based on an artificial neuron called _threshold logic unit_ (TLU) (also called _linear threshold unit_ (LTU).
- inputs and outputs are numbers.
- each input is associated with a weight.
- TLU computes the weighted sum of all these weights ($z = w_1x_1+w_2x_2+\dots+w_nx_n = \textbf{x}^T\textbf{w}$), then applies a step function on it.
- result: $h_w(\textbf{x}) = \text{step}(z)$ where $z = \textbf{x}^T\textbf{w}$

- common step functions in Perceptrons: $$ \begin{equation*}
\text{heaviside}(z) = 
\begin{cases} 
0 & \text{if } z < 0 \\ 
1 & \text{if } z \geq 0 
\end{cases}
\qquad
\operatorname{sgn}(z) = 
\begin{cases} 
-1 & \text{if } z < 0 \\ 
0 & \text{if } z = 0 \\ 
+1 & \text{if } z > 0 
\end{cases}
\end{equation*} $$
- the Heaviside step function is the most commonly used in Perceptrons.

- a single TLU can be used for simple linear binary classification.

- a Perceptron is composed of a single layer of TLUs, with each TLU connected to all the inputs.
- when all inputs in a layer are connected to every neuron in the previous layer, the layer is called _fully connected layer_ or a _dense_ layer.
- inputs of Perceptron are fed to special neurons called the _input layer_.
- an extra bias feature is added ($x_0 = 1$): it is represented using a special type of of neuron called a _bias neuron_, which outputs 1 all the time. 

- computing outputs of a fully connected layer: $h_{\textbf{W}, \textbf{b}}(\textbf{X}) = \phi(\textbf{X}\textbf{W}+\textbf{b})$
- where $\textbf{X}$ is matrix of input features. It has one row per instance and one column per feature.
- $\textbf{W}$ contains all connection weights except for the ones from the bias neuron. It has one row per input neuron and one column per artificial neuron in the layer.
- the bias vector $\textbf{b}$ contains all the connection weights between the bias neuron and the artificial neurons. It has one bias term per artificial neuron.
- the function $\phi$ is called _activation function_: when the artificial neurons are TLUs, it is a step function. 

- Perceptrons are trained using a variant of Hebb's rule that takes into account the error made by the network when it makes a prediction.
- the Perceptron is fed one training instance at a time and for each instance it makes a prediction. For every output neuron that produced a wrong prediction, it reinforces the connection weights from the inputs that would have contributed to the correct prediction.
- Perceptron learning rule (weight update): $$w_{i, j}^{(\text{next step})}=w_{i, j}+\eta(y_j-\hat{y}_j)x_i$$.
- $w_{i,j}$ is the connection weight between the $i^{\text{th}}$ input neuron and the $j^{\text{th}}$ output neuron.
- $x_i$ is the $i^{\text{th}}$ input value of the current training instance.
- $\hat{y}_j$ id the output of $j^{\text{th}}$ output neuron for the current training instance.
- $y_j$ is the target output of the $j^{\text{th}}$ output neuron for the current training instance.
- $\eta$ is the learning rate.

Perceptrons are incapable of learning complex patterns. However, if training instances are linearly separable, it can be demonstrated that this algorithm would converge to a solution. This is called _Perceptron convergence theorem_. 

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import Perceptron

iris = load_iris()
X = iris.data[:, (2,3)]
y = (iris.target == 0).astype(np.int32)

per_clf = Perceptron()
per_clf.fit(X, y)

,penalty,None
,alpha,0.0001
,l1_ratio,0.15
,fit_intercept,True
,max_iter,1000
,tol,0.001
,shuffle,True
,verbose,0
,eta0,1.0
,n_jobs,None
,random_state,0


In [2]:
y_pred = per_clf.predict([[2, 0.5]]) 

- Perceptron learning algorithm strongly resembles SGD. Scikit-Learn's `Perceptron` class is equivalent to using `SGDClassifier` with the following hyperparameters:
    - `loss="perceptron"`
    - `learning_rate="constant"`
    - `eta0=1`
    - `penalty=None`
- Perceptrons do not output a class probablity (unlike Logistic Regression), they make predictions based on a hard threshold. 

- Limitations of Perceptrons were noted in 1969 by Marvin Minsky and Seymour Papert in their monograph _Perceptrons_.
- They highlighted serious weaknesses of Perceptrons, in particular: the fact that they are incapable of solving trivial problems (like XOR classification problem).
- Some limitations of Perceptrons can be eliminated by stacking multiple Perceptrons. The resulting ANN is called a Multilayer Perceptron (MLP). 

## Multilayer Perceptron and Backpropagation

- An MLP is composed of one _input layer_, one or more layers of TLUs (called _hidden layers_, and one final layer of TLU's called the _output layer_.
- Layers closer to input layer are called _lower layers_, and the ones closer to outputs are called the _upper layers_.
- Every layer except the output layer includes a bias neuron and is fully connected to the next layer.
- When an ANN contains a deep stack of hidden layers, it is called a _deep neural network_ (DNN). Deep learning students DNNs.
- For many years, researchers struggled to find a way to train MLPs. In 1986, Geoffrey Hinton, David Rumelhard, and Ronald Williams published a groundbreaking paper which introduced the _backpropagation_ training algorithm.
- In short, it is Gradient Descent using an efficient technique for computing the gradients automatically: in just two passes through the network, the backpropagation algorithm is able to compute the gradient of the network's error with regard to every single model parameter. In other words, it can find out how each connection weight and each bias term should be tweaked in order to reduce the error. Once it has these gradients, it just performs a regular Gradient Descent step, and the whole process is repeated until the network converges to the solution.

**Note**: Automatically calculating gradients is called _automatic differentiation_ or _autodiff_. The autodiff technique that backpropagation uses is called _reverse-mode autodiff_. 

### Backpropagation Algorithm

1. It handled one mini-batch at a time and it goes through the full training set multiple times. Each pass is called an _epoch_.
2. Each mini-batch is passed to the NN's input layer, which sends it to the first hidden layer. The algorithm then computes the output of all the neurons in this layer. The result is passed on to the next layer, its output is computed and passed to the next layer, and so on until we get the output of the last layer: the output layer. This is called the _forward pass_: it is like making predictions, except all intermediate results are preserved since they are needed for the backward pass.
3. Next, the algorithm measures the network's output error (uses a loss function to compare desired and actual output and returns some measure of the error).
4. Then it computes how much each output connection contributed to the error. This is done analytically by applying the _chain rule_.
5. The algorithm measures how much of these error contributions came from each connection in the layer below, again using the chain rule, working backwards until the algorithm reaches the input layer. This reverse pass efficiently measures the error gradient across all the connection weights in the network by propagating the error gradient backword through the network. (Insight: how do you get the gradient for connections into B? You need $\partial Loss/\partial B_{\text{output}}$ first — but B doesn't touch the loss directly. B only affects the loss indirectly, by influencing C's output. That's the whole trick. If B's output feeds into multiple neurons in C (say C1, C2, C3), then B's activation influenced the loss through three different routes. So its total responsibility for the error is the sum of its contributions along every one of those routes.) $$ \frac {\partial \text{Loss}} {\partial B_{\text{output}}} = \sum_{i \in \text{C neurons}} {\frac {\partial \text{Loss}} {\partial C_{\text{i output}}} \times \frac {\partial C_\text{i output}} {\partial B_\text{output}}}$$
6. Finally, the algorithm performs a Gradient Descent step to tweak all the connection weights in the network, using the error gradients it just computed. 

In summary, for each training instance, the backpropagation algorithm first makes a prediction (forward pass) and measures the error, then goes through each layer in reverse to measure the error contribution from each connection (reverse pass), and finally tweaks the connection weights to reduce the error (Gradient Descent step).

- It is important to initialize all the hidden layers' connection weights randomly or the model will fail. If they're all initialized to the same number, the model would behave as if it had one neuron per layer.
- In order for this algorithm to work properly, its author made a key change to the MLP's architecture: they replaced the step function with the logistic (sigmoid) function, $\sigma(z) = \frac {1} {1 + \text{exp(-z)}}$.
- This was essential because the step function contains only flat segments, so there is no gradient to work with (Gradient Descent can't move on a flat surface), while the logistic function has a well-defined nonzero derivative everywhere, allowing GD to make some progress at every step.
-  Two more activation functions backpropagation works well with:
    - **The hyperbolic tangent function**: S-sshaped, continuous, and differentiable but it's output values ranges from -1 to 1 (instead of 0 to 1 in the case of the logistic function). This range helps to make each layer's output more centered around 0 at the beginning of training, which often helps speed up convergence. $$ \text{tanh}(z) = 2\sigma(2z)-1 $$ 
    - **The Rectified Linear Unit function:** ReLU function is coninuous but not differentiable at $z=0$. $$ \text{ReLU}(z) = \text{max}(0, z) $$

**Why do we need activations?**
If we chain several linear transformations, all we get is a linear transformation (e.g., let $f(x)$ and $g(x)$ be a linear function; $f(g(x))$ will also be a linear function). So, if we don't have some non-linearity between layers, then even a deep stack of layers is equivalent to a single layer. 

## Regression MLPs

- MLPs can be used for regression.
- We need one output neuron per output dimension (if more than one output, i.e., multivariate regression).
- In general, we don't want to use activation function for output neurons, so they are free to output any range of values. If we want to guarantee the output will be positive, then ReLU activation function can be used. Alternatively, _softplus_ activation function can be used, which is a smoother variant of ReLU. $$ \text{softplus}(z) = \text{log} (1 + \text{exp}(z)) $$
- It is close to 0 when $z$ is negative and close to $z$ when z is positive.
- If we want the output to fall within a set of values, we can use logistic function or hyperbolic tangent, and then sacel the labels to fit the range (0 to 1 and -1 to 1 respectively).
- Loss function used is typically mean squared error, but if there are a lot of outliers, mean absolute error maybe preferred. Alternatively, _Huber loss_ can be used which is a mix of both.

## Classification MLPs

- MLPs can also be used for classification.
- For binary classification, we only need one output neuron using the logisticactivation function (output will be between 0 and 1), which can be interpreted as estimated probablity of the positive class.
- MLPs can also handle multilabel binary classification. You would dedicate one neuron for each positive class. Note that output probablitites do not necessarily add up to 1.
- If each instance can belong only to a single class, out of three or more possible classes (e.g., classes 0 through 0 for digit classification), then we need to have one output neuron per class, and we should use softmax activation function for the whole output layer. The softmax function will ensure that all the estimated probablities are between 0 and 1 and that they add up to 1. This is called multiclass classification. 